# 02i — Re-diagnose the trajectory on the corrected ruler

**Accelerator: GPU T4 x2.** Estimated 2–3 h.

**Inputs to attach:**
- `02h-ruler-audit` — for `data/val.phonemes.jsonl` and `audit/budget_scale.json`
- `02-llm-finetune` — for `checkpoints/translation_llm/checkpoint-*`

## The question this session answers

Every number ever measured about this model was measured with a broken ruler. The labels
were non-space character counts (confirmed at 1.00 across all 53,350 rows), so the
"requested" budget in every evaluation was in one unit and the model's output was scored in
another. This session re-measures the existing checkpoints against **true phoneme counts**,
using the relabelled validation set.

Three things it settles:

1. **How much of the reported 10.3% length error was label noise rather than model error.**
   Per-row label error runs 5.7%–11.3% — comparable to the model's *entire* reported error.
   If the corrected numbers come out materially better with no change to the model, then
   the fine-tune was always better than its own evaluation said.

2. **Whether Assamese and Odia run short.** Relabelling moved their mean budgets **+16.5%**
   and **+15.2%** — the model was told "40" when the truth was 46. It should therefore
   under-produce in exactly those two languages. This is a falsifiable prediction: if `as`
   and `or` do *not* show the most negative signed adherence, the causal story is wrong and
   the diagnosis needs revisiting before anything is retrained.

3. **Whether 3,801 was actually early.** The stopping rule now runs as code against the
   *probe* slope — the estimator that holds the sentence fixed and varies only the budget —
   rather than the population slope that the old report quoted, which is confounded by
   sentence length and which a budget-ignoring model still scores well on.

## What this session is NOT

It is not a decision about whether to retrain. That decision is already made: the audit
returned RETRAIN. This produces the **baseline the retrained model will be compared
against**. Retraining first and comparing against a number measured with the old ruler
would repeat the exact error being repaired.


## 1. Dependencies and preflight

W&B needs `WANDB_API_KEY` as a Kaggle secret (Add-ons -> Secrets). If it is absent the run
still completes — the evaluation artifacts on disk are the source of truth and W&B is a
mirror, not a dependency.

In [ ]:
# W&B credential. This must succeed BEFORE anything expensive runs.
#
# Kaggle publishes a notebook's log only when the session ENDS, so W&B is the only channel
# that can be watched while a multi-hour run is happening. Session 02i proved what the
# lenient version costs: the secrets service returned a connection error, the notebook
# warned and carried on, and ~12 GPU-hours of a 30-hour weekly quota ran unobservable.
#
# The failure there was transient (service unreachable), not a missing secret — so retry
# before giving up, and accept a key that is already in the environment.
import os
import time

def _load_wandb_key(attempts: int = 3, pause: float = 5.0) -> str:
    if os.environ.get("WANDB_API_KEY"):
        return "environment"
    last = None
    for i in range(attempts):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
            return "kaggle-secrets"
        except Exception as e:  # noqa: BLE001
            last = e
            if i + 1 < attempts:
                time.sleep(pause)
    raise RuntimeError(
        "WANDB_API_KEY unavailable after {} attempts ({}).\n"
        "  Add-ons -> Secrets -> WANDB_API_KEY, and tick it for this notebook.\n"
        "  Refusing to start: Kaggle publishes the notebook log only when the session "
        "ENDS, so without W&B this run cannot be observed while it is happening."
        .format(attempts, last)
    )

_src = _load_wandb_key()
key = os.environ["WANDB_API_KEY"]
# Never print the value. A W&B key is 40 hex characters; anything else is the wrong string
# and is worth catching here rather than at wandb.init after the model is loaded.
print("WANDB_API_KEY loaded from {} ({} chars){}".format(
    _src, len(key), "" if len(key) == 40 else "  <-- expected 40, check the secret"))

os.environ.setdefault("WANDB_ENTITY", "nktthegreat-soccernet")
os.environ.setdefault("WANDB_PROJECT", "indic-dubbing-v3")


In [ ]:
# Record the platform's torch BEFORE installing anything. Kaggle's torch is built to match
# the driver and the T4's compute capability (sm_75). `pip install unsloth` will happily
# pull a newer torch whose binaries have no sm_75 kernels, and the failure is
# `CUDA error: no kernel image is available for execution on the device` — which appears
# only when the first kernel actually launches, long after the install "succeeded".
# requirements.txt warns about this in capitals; the second attempt at this notebook
# ignored it and lost a GPU session.
import torch
TORCH_BEFORE = torch.__version__
print("platform torch:", TORCH_BEFORE, "| device:", torch.cuda.get_device_name(0))
print("compute capability:", torch.cuda.get_device_capability(0))


In [ ]:
# Plain install, WITH dependencies. An earlier attempt used `--no-deps` to stop pip
# replacing torch — but torch was never the problem (it was a Tesla P100 assignment, see
# machine_shape), and `--no-deps` then left torchao pinned at 0.10.0 while transformers
# requires >0.16, which failed inside the eval subprocess. Solving a hypothetical risk
# created a real one. This is what the original 02_llm_finetune notebook used and what is
# known to work on a T4.
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip -q install unsloth 2>&1 | tail -2
!pip -q install phonemizer indic-nlp-library sacrebleu sentence-transformers wandb 2>&1 | tail -2


In [ ]:
# THE PREFLIGHT THAT MATTERS. Not "did pip succeed", not "is CUDA available" — both were
# true in the run that died. Three real checks, each of which fails in seconds:
#   1. torch is still the platform build (sm_75 kernels present)
#   2. a kernel actually launches on this device
#   3. the heavy import really resolves, which is where --no-deps gaps surface
import subprocess, torch

print(subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True).stdout.strip())
print(f"torch: {torch.__version__} (was {TORCH_BEFORE})")
if torch.__version__ != TORCH_BEFORE:
    print(f"NOTE: torch changed {TORCH_BEFORE} -> {torch.__version__}. Fine on a T4 (sm_75);"
          f" only sm_60 (P100) is unsupported by current builds.")
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 0), (
    f"GPU is compute capability {cap} ({torch.cuda.get_device_name(0)}). Current torch "
    f"builds ship sm_70+ only, so nothing will execute. Push with "
    f"machine_shape=NvidiaTeslaT4.")

x = torch.randn(64, 64, device="cuda")
assert torch.isfinite(x @ x).all().item(), "GPU matmul did not produce finite values"
torch.cuda.synchronize()
print("GPU kernel launch: OK")

from unsloth import FastLanguageModel          # the import that actually exercises the deps
import phonemizer, indicnlp, sacrebleu, sentence_transformers
print("imports OK: unsloth phonemizer indicnlp sacrebleu sentence_transformers")


## 2. Embedded code

Verbatim from the repo, so the run is reproducible on its own.

In [ ]:
import os, sys
for d in ("common", "translation", "evaluation"):
    os.makedirs(f"/kaggle/working/pipeline_v3/{d}", exist_ok=True)
open("/kaggle/working/pipeline_v3/evaluation/__init__.py", "w").close()
os.chdir("/kaggle/working/pipeline_v3")
sys.path.insert(0, "/kaggle/working/pipeline_v3")
print(os.getcwd())


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/__init__.py
"""Shared utilities used across translation/, training/, and tts/: language metadata
(languages.py) and CTC forced alignment (forced_alignment.py)."""


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/languages.py
"""
common/languages.py
--------------------
Single source of truth for language metadata used across pipeline_v3.

Every other module (dataset_generator, duration_predictor, isochrony_translation_v3,
train_translation_llm, data_augmentation, train_tts) imports from here instead of
hard-coding its own language table. This avoids the classic bug where "hi" is spelled
one way in one file and another way in another file.

The 11 languages below are the intersection of:
  - Samanantar's 11 Indic targets (as, bn, gu, hi, kn, ml, mr, or, pa, ta, te)
  - IndicF5's 11 supported languages (same set)
  - espeak-ng's Indic voice coverage (verified: all 11 present as of espeak-ng 1.51)

NOTE ON EXPANSION RATIOS: The `heuristic_expansion_ratio` and `heuristic_phonemes_per_sec`
values below are *cold-start defaults*, carried forward from/consistent with the existing
V2 pipeline's phoneme_counter.py approach (e.g. Hindi=1.30, Tamil=1.35). They are
deliberately approximate. The entire point of V3 is to replace these hard-coded numbers
with the learned DurationPredictor (see translation/duration_predictor.py) once you have
trained it on real forced-aligned speech. Treat this table as the fallback that is used
(a) before you've trained anything, and (b) as a sanity-check ceiling/floor on predicted
durations even after training.
"""

from dataclasses import dataclass


@dataclass(frozen=True)
class LanguageInfo:
    name: str                          # Human-readable display name
    iso_code: str                      # 2-letter ISO 639-1 code, used by Samanantar & IndicF5
    espeak_code: str                   # espeak-ng voice code (for phonemizer backend="espeak")
    indicf5_code: str                  # Code IndicF5 expects (matches iso_code for all 11)
    script: str                        # Unicode script name, for sanity checks / logging
    heuristic_expansion_ratio: float   # Indic-text-length / English-text-length, rough prior
    heuristic_phonemes_per_sec: float  # Cold-start speaking rate for DurationPredictor fallback


# fmt: off
LANGUAGES: dict[str, LanguageInfo] = {
    "hi": LanguageInfo("Hindi",      "hi", "hi", "hi", "Devanagari", 1.30, 13.5),
    "bn": LanguageInfo("Bengali",    "bn", "bn", "bn", "Bengali",    1.28, 13.0),
    "mr": LanguageInfo("Marathi",    "mr", "mr", "mr", "Devanagari", 1.27, 13.2),
    "gu": LanguageInfo("Gujarati",   "gu", "gu", "gu", "Gujarati",   1.25, 13.0),
    "pa": LanguageInfo("Punjabi",    "pa", "pa", "pa", "Gurmukhi",   1.22, 12.8),
    "ta": LanguageInfo("Tamil",      "ta", "ta", "ta", "Tamil",      1.35, 12.5),
    "te": LanguageInfo("Telugu",     "te", "te", "te", "Telugu",     1.33, 12.6),
    "kn": LanguageInfo("Kannada",    "kn", "kn", "kn", "Kannada",    1.30, 12.7),
    "ml": LanguageInfo("Malayalam",  "ml", "ml", "ml", "Malayalam",  1.38, 12.3),
    "or": LanguageInfo("Odia",       "or", "or", "or", "Odia",       1.26, 13.0),
    "as": LanguageInfo("Assamese",   "as", "as", "as", "Bengali",    1.27, 12.9),
}
# fmt: on

# Samanantar's HF dataset config names are lowercase 2-letter codes identical to iso_code,
# EXCEPT the source side, which is always English.
SAMANANTAR_SOURCE_LANG = "en"
SAMANANTAR_HF_PATH = "ai4bharat/samanantar"

# ai4bharat/Kathbath directory structure uses full language *names* (lowercase), not codes.
KATHBATH_DIR_NAMES: dict[str, str] = {
    "hi": "hindi", "bn": "bengali", "mr": "marathi", "gu": "gujarati",
    "pa": "punjabi", "ta": "tamil", "te": "telugu", "kn": "kannada",
    "ml": "malayalam", "or": "odia", "as": "assamese",
    # Kathbath additionally has Sanskrit/Urdu/Nepali/Chhattisgarhi splits not covered here
    # because they fall outside the 11-language Samanantar/IndicF5 intersection.
}
KATHBATH_HF_PATH = "ai4bharat/Kathbath"

# ai4bharat/Rasa is the (much closer to IndicF5's own training mix) TTS-quality speech
# dataset: studio recordings, 13 languages, ~500+ hours, MIT-tagged on HF. Prefer this over
# Kathbath (which is ASR-oriented, phone/field recordings) for anything TTS-related, and
# fall back to Kathbath only if you need more raw hours than Rasa provides for a language.
RASA_HF_PATH = "ai4bharat/Rasa"

# ai4bharat/BPCC (Bharat Parallel Corpus Collection) - the successor to Samanantar:
# ~230M pairs, 22 languages, and (unlike standalone Samanantar's ambiguous HF license
# tag) an EXPLICIT license table on its dataset card: all MINED corpora - including
# Samanantar (19.4M) and Samanantar++ (121.6M) - are CC0; the human-annotated seed
# subsets (BPCC-H-Wiki/Daily) are CC-BY-4.0. This resolves the Samanantar license
# ambiguity flagged in dataset_generator.py for commercial use. GATED: accept the
# conditions on huggingface.co/datasets/ai4bharat/BPCC with your HF account first.
#
# STRUCTURE (verified 2026-07-13 by browsing the gated repo directly, including
# AI4Bharat's own compile.py in the repo root, which generated these files):
# BPCC is a RAW-FILE repo, not a config-per-language dataset - the HF dataset viewer is
# disabled for it and it has no loading script, so load_dataset("ai4bharat/BPCC",
# <config>) does NOT work. The data lives in per-subset directories of per-language
# TSVs keyed by FLORES-200 code:
#     <subset>/<flores_code>.tsv    e.g.  samanantar_v2/hin_Deva.tsv  (6.9 GB)
# Each TSV is tab-separated WITH a header row and exactly these columns (from
# compile.py: df.to_csv(sep='\t', index=False)):
#     src_lang  tgt_lang  src  tgt      (src = English text, tgt = Indic text)
# All 11 of our languages are present in samanantar_v2/ (plus npi/urd, unused here).
# dataset_generator.py streams these via load_dataset("csv", data_files="hf://...").
# Subsets seen in the repo: samanantar_v2 (default - 28.9GB, the filtered mined set),
# samanantar_v0.3_filtered, nllb_filtered, nllb_seed, bpcc-seed-latest/v1/v2,
# comparable, daily, ilci, massive, wiki. Only samanantar_v2's internal layout was
# inspected; treat other subsets' layouts as unverified until probed.
BPCC_HF_PATH = "ai4bharat/BPCC"
BPCC_SOURCE_FLORES = "eng_Latn"
BPCC_DEFAULT_SUBSET = "samanantar_v2"
BPCC_FLORES_CODES: dict[str, str] = {
    "hi": "hin_Deva", "bn": "ben_Beng", "mr": "mar_Deva", "gu": "guj_Gujr",
    "pa": "pan_Guru", "ta": "tam_Taml", "te": "tel_Telu", "kn": "kan_Knda",
    "ml": "mal_Mlym", "or": "ory_Orya", "as": "asm_Beng",
}

# IndicF5 itself
INDICF5_HF_PATH = "ai4bharat/IndicF5"
INDICF5_SAMPLE_RATE = 24000

# ai4bharat/vits_rasa_13 - fixed-inventory multi-speaker VITS TTS (40.2M params,
# CC-BY-4.0, GATED). Everything below is transcribed 1:1 from the model card's own
# speaker/style tables (verified 2026-07-13 via authenticated access, not assumed).
# IMPORTANT COVERAGE FACT: the model supports 13 languages but NOT Hindi, Gujarati, or
# Odia - three of this pipeline's 11 targets. hi/gu/or must use the IndicF5 backend
# for multi-speaker dubbing (see tts/vits_rasa_tts.py and multi_speaker_dubbing.py).
VITS_RASA_HF_PATH = "ai4bharat/vits_rasa_13"
VITS_RASA_SPEAKERS: dict[str, int] = {
    "ASM_F": 0, "ASM_M": 1, "BEN_F": 2, "BEN_M": 3, "BRX_F": 4, "BRX_M": 5,
    "DOI_F": 6, "DOI_M": 7, "KAN_F": 8, "KAN_M": 9, "MAI_M": 10, "MAL_F": 11,
    "MAR_F": 12, "MAR_M": 13, "NEP_F": 14, "PAN_F": 15, "PAN_M": 16, "SAN_M": 17,
    "TAM_F": 18, "TEL_F": 19,
}
# Style/emotion IDs exactly as published (note the real gaps at 9/11/13 - the model
# card skips those IDs; do not "fix" this by renumbering).
VITS_RASA_STYLES: dict[str, int] = {
    "ALEXA": 0, "ANGER": 1, "BB": 2, "BOOK": 3, "CONV": 4, "DIGI": 5, "DISGUST": 6,
    "FEAR": 7, "HAPPY": 8, "NEWS": 10, "SAD": 12, "SURPRISE": 14, "UMANG": 15, "WIKI": 16,
}
# Per-target-language voice availability, keyed by this pipeline's ISO codes.
# None = that gender doesn't exist in the model (ml/ta/te are female-only).
VITS_RASA_VOICES_BY_LANG: dict[str, dict] = {
    "as": {"F": 0, "M": 1},
    "bn": {"F": 2, "M": 3},
    "kn": {"F": 8, "M": 9},
    "ml": {"F": 11, "M": None},
    "mr": {"F": 12, "M": 13},
    "pa": {"F": 15, "M": 16},
    "ta": {"F": 18, "M": None},
    "te": {"F": 19, "M": None},
}
VITS_RASA_UNSUPPORTED: frozenset = frozenset({"hi", "gu", "or"})

# pyannote speaker diarization (multi-speaker dubbing's segmentation stage). BOTH repos
# are gated on HF - accept terms on each model page with the same HF_TOKEN account:
#   huggingface.co/pyannote/speaker-diarization-3.1
#   huggingface.co/pyannote/segmentation-3.0  (pulled in by the pipeline above)
PYANNOTE_DIARIZATION_HF_PATH = "pyannote/speaker-diarization-3.1"


def get_language(code: str) -> LanguageInfo:
    code = code.lower().strip()
    if code not in LANGUAGES:
        raise ValueError(
            f"Unknown language code '{code}'. Supported: {sorted(LANGUAGES.keys())}"
        )
    return LANGUAGES[code]


def all_codes() -> list[str]:
    return list(LANGUAGES.keys())


def display_name(code: str) -> str:
    return get_language(code).name


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/phonemes.py
"""
common/phonemes.py
==================
THE canonical grapheme-to-phoneme counter for pipeline_v3. Every module that writes a
phoneme budget into a label, and every module that scores a generation against one, must
import `count_phonemes` from here and from nowhere else.

WHY THIS MODULE EXISTS (read this before changing anything in it)
-----------------------------------------------------------------
The previous implementation lived in `translation/duration_predictor.phonemize_text` and
ended like this::

    except Exception as e:
        logger.debug("Phonemization failed ...; falling back to characters.")
    return [c for c in text if not c.isspace()]

That fallback fired for the **entire** corpus generation run, because espeak-ng (a system
binary, installed separately from the `phonemizer` pip package) was not present in that
Kaggle session. It logged at DEBUG, so nothing surfaced. The result: every `n_phonemes`
label in the base training corpus is a **non-space character count**, verified at 100.0%
over 1,289 locally-held rows with a chars/n_phonemes ratio of exactly 1.000 (min = max).

It got worse. A later session (length augmentation, 2026-07-26) *did* have espeak-ng, so
those rows are labelled in **real phonemes** — 8.4% coincidental agreement with character
counts, ratio spread 0.198-1.500. The corpus therefore carries two mutually incompatible
rulers under one prompt token, `[Target Phonemes: N]`, teaching the model two
contradictory tasks. A model asked to fit N of one unit and N of another cannot reach
slope 1.0 on either; it can only split the difference.

So this module enforces three things the old one did not:

1. **No silent fallback, ever.** A phonemization failure raises `G2PUnavailable` or
   `PhonemizationError`. Label-writing code must never degrade quietly, because a
   degraded label is indistinguishable from a good one downstream. Inference code that
   legitimately needs to survive a bad string catches the exception *explicitly* and
   records the degradation (see "degrade, don't crash" — but degrade *visibly*).

2. **A preflight that proves the output is phonemes, not passthrough.** Checking that
   espeak-ng is importable is not sufficient — the failure mode we actually hit produces
   plausible-looking output. `assert_g2p_available()` phonemizes a canary string in each
   language and asserts the returned symbols are not simply the input's own characters.
   That is the check that would have caught this on day one.

3. **A ruler identifier stamped into every artifact.** `ruler_id()` returns a string like
   ``phonemes:espeak-ng-1.51``. Dataset rows, eval reports, and run manifests all carry
   it, so "which ruler produced this number" is a grep, not a forensic exercise.

WHAT COUNTS AS ONE PHONEME
---------------------------
espeak-ng's IPA output carries symbols that are not sounds. We normalise before counting:

- **Stress marks** (``ˈ`` primary, ``ˌ`` secondary) are suprasegmental — they mark which
  syllable is emphasised, not an additional sound. Stripped.
- **Length marks** (``ː``) modify the preceding vowel's duration and stay attached to it,
  so ``aː`` is one (long) phoneme, not two. Kept attached — which is correct for our
  purpose, since duration is exactly what we are proxying.
- **Language-switch tags** (``(en)``, ``(hi)``) are emitted when espeak detects a foreign
  word — typically English brand names inside Indic text. They are markup. Stripped.
- **Tie bars** (``͡``) join affricates into one segment. Kept attached.

These choices are asymmetric-safe: they can only ever be wrong by a constant per language,
and non-negotiable #3 (same function for labels and scores) means a constant offset
cancels. What must never happen is two *different* normalisations in the same project.
"""

from __future__ import annotations

import functools
import logging
import re
from typing import Iterable, Optional, Sequence

from common.languages import LANGUAGES, get_language

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------------------
# Ruler identifiers. These are written into datasets and reports; treat them as a stable
# public vocabulary, not as free text.
# ---------------------------------------------------------------------------------------

RULER_PHONEMES = "phonemes:espeak-ng"
RULER_CHARS = "chars:non-space"
RULER_UNKNOWN = "unknown"


class G2PUnavailable(RuntimeError):
    """espeak-ng and/or the `phonemizer` package is missing or non-functional.

    Raised by the preflight and by `phonemize()`. This is deliberately fatal: every
    caller in the label-writing path would otherwise produce a corpus that looks correct
    and is measured in the wrong unit.
    """


class PhonemizationError(RuntimeError):
    """espeak-ng is present and working, but this specific string could not be converted."""


_INSTALL_HINT = (
    "espeak-ng is a SYSTEM package and is NOT installed by `pip install phonemizer`.\n"
    "  Kaggle / Debian / Ubuntu:  apt-get -qq update && apt-get -qq install -y espeak-ng\n"
    "  macOS:                     brew install espeak-ng\n"
    "  Windows:                   winget install --id eSpeak-NG.eSpeak-NG   (then set\n"
    "                             PHONEMIZER_ESPEAK_LIBRARY to the installed libespeak-ng.dll)\n"
    "  Then:                      pip install phonemizer\n"
    "Verify with: python -c \"from common.phonemes import assert_g2p_available;"
    " assert_g2p_available()\""
)

# Suprasegmental / markup symbols that are not themselves sounds.
_STRESS_MARKS = "ˈˌ"          # ˈ ˌ
_LANG_SWITCH_RE = re.compile(r"\([a-z]{2,3}\)")   # (en), (hi), ...
_UNDERTIE = "‿"

# Unicode names several scripts by an older label than the one the language table uses.
_UNICODE_SCRIPT_NAME = {"odia": "ORIYA"}


def _script_token(language_iso_code: str) -> str:
    """The token that appears in unicodedata.name() for this language's script."""
    script = get_language(language_iso_code).script.lower()
    return _UNICODE_SCRIPT_NAME.get(script, script.split()[0].upper())


# ---------------------------------------------------------------------------------------
# Orthographic normalisation (AI4Bharat IndicNLP)
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=16)
def _normalizer(language_iso_code: str):
    """Per-language IndicNLP normaliser, or None if the library is absent."""
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    except ImportError:
        return None
    return IndicNormalizerFactory().get_normalizer(language_iso_code)


# Set once, the first time normalisation is skipped, so the absence is stated rather than
# inferred from a slightly-off number three stages downstream.
_NORMALIZER_WARNED: set = set()


@functools.lru_cache(maxsize=1)
def _recomposition_map() -> dict:
    """decomposed-sequence -> precomposed-character, for Indic nukta letters.

    Built from `unicodedata` rather than hand-listed, so it cannot drift from the standard
    and needs no maintenance when a script is added.

    WHY IT IS NEEDED. IndicNLP canonicalises *toward the decomposed form*: `য়` U+09DF
    becomes U+09AF + U+09BC (YA + NUKTA). espeak-ng's rule files are evidently written
    against the *precomposed* letters. Measured on this corpus: normalisation rewrites
    50.8% of Assamese rows, almost all of them this substitution, and Assamese
    phonemes-per-character then jumps 1.147 -> 1.604 while Bengali — same script, same
    substitution on 36.3% of its rows — barely moves (1.032 -> 1.020). Assamese and Bengali
    are phonologically close and orthographically shared; a 57% divergence between them is
    not a better measurement, it is espeak's `as` voice failing to parse a sequence its
    `bn` voice handles.

    These characters cannot be recomposed by `unicodedata.normalize("NFC", ...)`, because
    Indic nukta letters are Unicode *composition exclusions* — NFC deliberately leaves them
    decomposed. Hence an explicit map.

    So: take IndicNLP's genuine cleanups (ZWJ/ZWNJ removal, punctuation canonicalisation,
    Malayalam chillu handling) and then put the nukta letters back into the encoding the
    G2P actually recognises.
    """
    import unicodedata
    mapping = {}
    # Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu, Kannada, Malayalam
    for cp in range(0x0900, 0x0E00):
        ch = chr(cp)
        decomp = unicodedata.decomposition(ch)
        if not decomp or decomp.startswith("<"):   # skip compatibility decompositions
            continue
        try:
            seq = "".join(chr(int(p, 16)) for p in decomp.split())
        except ValueError:
            continue
        mapping[seq] = ch
    return mapping


def recompose_indic(text: str) -> str:
    """Restores precomposed Indic nukta letters after IndicNLP normalisation."""
    for seq, ch in _recomposition_map().items():
        if seq in text:
            text = text.replace(seq, ch)
    return text


def normalize_indic(text: str, language_iso_code: str) -> str:
    """Canonicalises Indic text before G2P.

    Indic scripts encode the same grapheme several ways, and espeak-ng only has rules for
    one of them. Verified on this machine:

        क़  U+0958 (precomposed)      -> U+0915 U+093C  (KA + NUKTA)
        ড়  U+09DC (precomposed)      -> U+09A1 U+09BC  (DDA + NUKTA)
        क्‍ष  with U+200D ZWJ            -> ZWJ removed

    Fed the precomposed or ZWJ-bearing form, espeak either skips the codepoint or emits
    something arbitrary — silently, and only for the subset of rows that happen to use that
    encoding. That is a per-row error concentrated in exactly the words most likely to be
    loanwords and proper nouns, which is worse than a uniform bias because it cannot be
    calibrated away.

    Normalising is a strict improvement and costs nothing: on already-canonical text it is
    a no-op (verified across all 11 languages). It runs inside `phonemize`, so labels and
    scores are normalised identically — the same-function rule (non-negotiable #3) extends
    to preprocessing, not just to the G2P call.
    """
    n = _normalizer(language_iso_code)
    if n is None:
        if "warned" not in _NORMALIZER_WARNED:
            _NORMALIZER_WARNED.add("warned")
            logger.warning(
                "indic-nlp-library is not installed — phoneme counts will be taken on "
                "UN-normalised text. Precomposed nukta forms and ZWJ sequences will be "
                "mis-phonemized for a minority of rows. Install with: "
                "pip install indic-nlp-library"
            )
        return text
    return recompose_indic(n.normalize(text))


# ---------------------------------------------------------------------------------------
# Backend access
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=1)
def _backend_version() -> str:
    """Returns the espeak-ng version string, or raises G2PUnavailable.

    Cached because it shells into the espeak library, and callers ask for it once per
    artifact written.
    """
    try:
        from phonemizer.backend.espeak.wrapper import EspeakWrapper
    except ImportError as e:
        raise G2PUnavailable(f"`phonemizer` is not importable ({e}).\n{_INSTALL_HINT}") from e

    try:
        version = EspeakWrapper().version
    except Exception as e:  # noqa: BLE001 - any failure here means the shared library is unusable
        raise G2PUnavailable(
            f"`phonemizer` imported but the espeak-ng shared library is unusable ({e}).\n"
            f"{_INSTALL_HINT}"
        ) from e

    # Normalise: some phonemizer builds return a tuple like (1, 50) rather than "1.50".
    # `str()` on that yields "(1, 50)" — which embeds a COMMA and a space into a string
    # that gets stamped into every dataset row and every report header, and would corrupt
    # any CSV column it ever lands in. A provenance field that can break its own container
    # is not provenance.
    if isinstance(version, (tuple, list)):
        version = ".".join(str(p) for p in version)
    return re.sub(r"[\s,]+", "", str(version))


def ruler_id() -> str:
    """The identifier stamped into every dataset row and eval report this module touches.

    Raises G2PUnavailable rather than returning a placeholder — a report that cannot name
    its ruler must not be written at all.
    """
    return f"{RULER_PHONEMES}-{_backend_version()}"


# ---------------------------------------------------------------------------------------
# Core conversion
# ---------------------------------------------------------------------------------------

def _normalise_tokens(raw: str) -> list[str]:
    """Turns espeak's separated output into a list of countable phoneme symbols."""
    raw = _LANG_SWITCH_RE.sub(" ", raw)
    raw = raw.replace("|", " ").replace(_UNDERTIE, " ")
    tokens = []
    for tok in raw.split():
        tok = tok.strip(_STRESS_MARKS)
        # A token that was *only* stress marks collapses to empty and is not a sound.
        if tok:
            tokens.append(tok)
    return tokens


def phonemize(text: str, language_iso_code: str) -> list[str]:
    """Converts `text` to a list of IPA phoneme symbols. Never falls back.

    Raises:
        G2PUnavailable: espeak-ng / phonemizer is missing or broken.
        PhonemizationError: the backend works but produced nothing for this input.
    """
    text = (text or "").strip()
    if not text:
        return []

    _backend_version()  # raises G2PUnavailable with the install hint if the backend is dead

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    text = normalize_indic(text, language_iso_code)
    try:
        out = _ph(
            text,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on {language_iso_code} input {text[:60]!r}: {e}"
        ) from e

    tokens = _normalise_tokens(out)
    if not tokens:
        raise PhonemizationError(
            f"espeak-ng returned no phonemes for {language_iso_code} input {text[:60]!r}. "
            f"Raw backend output was {out[:80]!r}."
        )
    return tokens


def phonemize_many(texts: Sequence[str], language_iso_code: str) -> list[list[str]]:
    """Batched `phonemize`, one espeak call for the whole list.

    Roughly an order of magnitude faster than looping — which matters, because relabelling
    the 53,350-row corpus one string at a time is a multi-hour job and a batched pass is
    minutes. Empty inputs map to empty lists; a backend failure on the batch raises, so a
    partially-phonemized corpus is never written.
    """
    if not texts:
        return []

    _backend_version()

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    cleaned = [normalize_indic((t or "").strip(), language_iso_code) for t in texts]
    try:
        out = _ph(
            cleaned,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on a batch of {len(texts)} {language_iso_code} strings: {e}"
        ) from e

    if isinstance(out, str):  # phonemizer collapses a 1-element list to a bare string
        out = [out]
    return [_normalise_tokens(o) for o in out]


@functools.lru_cache(maxsize=200_000)
def count_phonemes(text: str, language_iso_code: str) -> int:
    """The one function that defines "how many phonemes is this". Labels and scores both
    call it, which is what makes their numbers comparable (non-negotiable #3)."""
    return len(phonemize(text, language_iso_code))


def phoneme_inventory(texts: Iterable[str], language_iso_code: str) -> "Counter":
    """The distribution of phoneme symbols a language's G2P actually produces.

    This is a *validation* instrument, not a pipeline metric — the pipeline only ever needs
    the per-sentence count. But a count cannot tell you whether the symbols being counted
    are phonemes at all, and an inventory can, in one glance.

    Worked example of what it catches: epitran's Marathi renders `कॅल्शियम` as `kəॅlɕijmə`,
    leaking U+0945 DEVANAGARI VOWEL SIGN CANDRA E — a *source script* character — straight
    into its own IPA output, because that codepoint has no entry in its map. The count still
    comes out looking reasonable. The inventory makes it obvious.
    """
    from collections import Counter
    inv = Counter()
    for toks in phonemize_many(list(texts), language_iso_code):
        inv.update(toks)
    return inv


def validate_inventory(inventory: "Counter", language_iso_code: str) -> list[str]:
    """Returns a list of problems found in a phoneme inventory; empty means clean.

    The check that matters: a symbol containing a character from the language's OWN script
    is not a phoneme — it is an unmapped source character that the G2P passed through
    untranslated. Its presence proves the converter has a hole, and tells you exactly which
    grapheme fell in.
    """
    import unicodedata

    token = _script_token(language_iso_code)
    problems = []
    total = sum(inventory.values()) or 1
    for sym, n in inventory.most_common():
        leaked = [c for c in sym if token in unicodedata.name(c, "")]
        if leaked:
            names = ", ".join(f"U+{ord(c):04X} {unicodedata.name(c, '?')}" for c in leaked)
            problems.append(
                f"{sym!r} ({n} occurrences, {100 * n / total:.2f}%) contains untranslated "
                f"source-script characters: {names}"
            )
    return problems


def count_chars(text: str) -> int:
    """Non-space character count — the *wrong* ruler, defined here explicitly so audit
    code can name it and detect it rather than reimplementing it three times."""
    return len([c for c in (text or "") if not c.isspace()])


# ---------------------------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------------------------

# Short canaries in each language's own script. Any real sentence works; these are kept
# tiny so the preflight costs milliseconds.
_CANARIES: dict[str, str] = {
    "hi": "दूध में कैल्शियम है",
    "bn": "দুধে ক্যালসিয়াম আছে",
    "mr": "दुधात कॅल्शियम आहे",
    "gu": "દૂધમાં કેલ્શિયમ છે",
    "pa": "ਦੁੱਧ ਵਿੱਚ ਕੈਲਸ਼ੀਅਮ ਹੈ",
    "ta": "பாலில் கால்சியம் உள்ளது",
    "te": "పాలలో కాల్షియం ఉంది",
    "kn": "ಹಾಲಿನಲ್ಲಿ ಕ್ಯಾಲ್ಸಿಯಂ ಇದೆ",
    "ml": "പാലിൽ കാൽസ്യം ഉണ്ട്",
    "or": "ଦୁଧରେ କ୍ୟାଲସିୟମ ଅଛି",
    "as": "গাখীৰত কেলচিয়াম আছে",
}


def assert_g2p_available(languages: Optional[Iterable[str]] = None, verbose: bool = True) -> dict:
    """Preflight. Call this at the top of every notebook that writes labels or scores them.

    Checks three things, in increasing order of strictness:

    1. `phonemizer` imports and the espeak-ng shared library loads.
    2. Every requested language produces non-empty output.
    3. **The output is actually phonemes, not the input's own characters.** This is the
       check that matters. A backend that silently passes text through, or a caller that
       silently substitutes a character split, both produce plausible non-empty output —
       and that is precisely the failure that mislabelled this project's corpus. We assert
       that at least one returned symbol does not appear in the source string, which is
       guaranteed true for any Indic script rendered to IPA and false for passthrough.

    Returns a manifest dict suitable for writing next to any artifact produced afterwards.

    Raises:
        G2PUnavailable: with the install hint, if any check fails.
    """
    codes = list(languages) if languages is not None else list(LANGUAGES.keys())
    version = _backend_version()   # raises with install hint

    report: dict[str, dict] = {}
    failures: list[str] = []

    for code in codes:
        canary = _CANARIES.get(code)
        if canary is None:
            failures.append(f"{code}: no canary string defined in common/phonemes.py")
            continue
        try:
            tokens = phonemize(canary, code)
        except (G2PUnavailable, PhonemizationError) as e:
            failures.append(f"{code}: {e}")
            continue

        source_chars = set(canary)
        novel = [t for t in tokens if not set(t) <= source_chars]
        looks_like_passthrough = not novel

        # Partial leaks: individual unmapped graphemes riding through into the IPA. The
        # passthrough test above only catches total failure; this catches the holes.
        from collections import Counter
        leaks = validate_inventory(Counter(tokens), code)

        report[code] = {
            "voice": get_language(code).espeak_code,
            "n_phonemes": len(tokens),
            "n_chars": count_chars(canary),
            "sample": " ".join(tokens[:12]),
            "passthrough": looks_like_passthrough,
            "normalized": _normalizer(code) is not None,
            "leaks": leaks,
        }
        if looks_like_passthrough:
            failures.append(
                f"{code}: espeak returned symbols drawn entirely from the input's own "
                f"characters ({' '.join(tokens[:10])}) — this is a passthrough or a "
                f"character-split fallback, NOT phonemization."
            )
        for lk in leaks:
            failures.append(f"{code}: untranslated grapheme in G2P output — {lk}")

    if failures:
        raise G2PUnavailable(
            "G2P preflight FAILED — do not generate labels or scores in this session.\n"
            + "\n".join(f"  - {f}" for f in failures)
            + "\n\n" + _INSTALL_HINT
        )

    manifest = {
        "ruler": ruler_id(),
        "espeak_version": version,
        "languages": report,
    }
    if verbose:
        logger.info("G2P preflight PASSED — ruler=%s", manifest["ruler"])
        for code, r in report.items():
            logger.info(
                "  %-3s voice=%-3s %3d phonemes / %3d chars (ratio %.3f)  %s",
                code, r["voice"], r["n_phonemes"], r["n_chars"],
                r["n_phonemes"] / max(r["n_chars"], 1), r["sample"],
            )
    return manifest


if __name__ == "__main__":  # `python -m common.phonemes` as a standalone preflight
    import json
    import sys

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        print(json.dumps(assert_g2p_available(), ensure_ascii=False, indent=2))
    except G2PUnavailable as e:
        print(str(e), file=sys.stderr)
        raise SystemExit(1)


In [ ]:
%%writefile /kaggle/working/pipeline_v3/translation/__init__.py
"""
translation/__init__.py
--------------------------
"""


In [ ]:
%%writefile /kaggle/working/pipeline_v3/translation/semantic_gate.py
"""
translation/semantic_gate.py
-----------------------------------------
Inference-time meaning-preservation check for isochrony-constrained translation.

WHY THIS EXISTS
---------------
`isochrony_translation_v3.generate_and_select` produces several candidate translations at
different phoneme budgets ("direct" 1.0, "paraphrase" 0.85, "minimal" 0.65) and, when
none of them fit the segment's time window, refines by tightening the budget a further
0.75x per round for up to 3 rounds. A segment can therefore legitimately end up at
0.65 * 0.75^3 ~= 0.27 of the original phoneme budget.

Until this module existed, the winning candidate was chosen by `_score_candidate`, which
looks *only* at predicted duration versus target duration. Nothing anywhere in the
inference path asked whether the surviving text still meant the same thing.

The precise failure this creates is worth stating carefully, because the obvious version
of the criticism is wrong. `_score_candidate` does NOT simply prefer the shortest text —
it scores closeness to the target duration, so a wildly over-compressed candidate is
penalised for undershooting just as an over-long one is penalised for overshooting.

The real gap is subtler and worse: **two candidates of identical predicted duration score
identically, whether one preserved the meaning and the other dropped a clause.** Hitting
a phoneme budget by rephrasing more tersely and hitting it by deleting the subordinate
clause produce the same number of phonemes, therefore the same predicted milliseconds,
therefore the same score. Timing is simply blind to the distinction. When the budget is
tight enough that faithful compression cannot fit, the deleting variant is the one that
lands on target — and it wins on merit under the old scoring.

That is precisely the failure mode the *training* pipeline already defends against:
`training/length_augmentation.py` gates every generated paraphrase on cosine similarity
>= 0.80 against the human reference, explicitly "to stop the model teaching itself
'delete random words to hit a phoneme count'". This module carries the same guarantee
into inference, where there is no human reference to compare against.

CHOICE OF ANCHOR (the part that is easy to get wrong)
-----------------------------------------------------
At training time the anchor is the human reference translation. At inference no reference
exists, so we need a stand-in for "what this segment is supposed to mean". Two are
available and we use both, in order of preference:

  1. SAME-LANGUAGE ANCHOR (preferred): the "direct" candidate, generated at the full 1.0
     phoneme budget. It is the least-compressed translation we have and it is produced
     anyway, so it costs nothing extra. Comparing a compressed candidate against it
     measures exactly what we care about: "how much meaning did compression cost?"
     Same-language cosines are well calibrated and directly comparable to the training
     gate's 0.80 threshold.

  2. CROSS-LINGUAL FALLBACK: the English source. Used when the direct candidate failed to
     generate or was empty. The embedder is multilingual and aligns translations across
     languages, but cross-lingual cosines sit systematically lower than same-language
     ones for identical meaning, so this path uses its own (lower) threshold. Do not
     compare the two numbers to each other.

The embedder is deliberately the SAME model the training gate uses
(paraphrase-multilingual-MiniLM-L12-v2). If the two used different embedders, the
threshold learned/validated on one would not transfer to the other.

FAILURE POLICY
--------------
This gate never crashes a dub and never silently ships a bad line:
  - If `sentence-transformers` is not installed, or the model cannot load, the gate
    disables itself, logs ONE warning, and selection falls back to timing-only scoring
    (exactly the old behaviour). Semantic scores are reported as None, not as 1.0, so
    downstream logs can tell "not checked" apart from "checked and fine".
  - If every candidate scores below threshold, the pipeline does not fail. The candidate
    with the highest similarity among those that fit the window is chosen and the segment
    is flagged `semantic_degraded=True`. A dub with one weak line is recoverable; a
    crashed 40-minute render is not. The flag is what makes the weak line findable.
"""

from __future__ import annotations

import logging
import os
from typing import Optional, Sequence

logger = logging.getLogger(__name__)

# Must match training/length_augmentation.py — see module docstring.
DEFAULT_EMBEDDER_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Same-language floor, aligned with the training gate's min_similarity=0.80.
DEFAULT_MIN_SIMILARITY_SAME_LANG = float(os.getenv("ISOCHRONY_MIN_SIM_SAME_LANG", "0.80"))

# Cross-lingual floor. Lower by construction: EN<->Indic cosines for identical meaning
# typically land ~0.60-0.75 with this embedder, well below same-language equivalents.
DEFAULT_MIN_SIMILARITY_CROSS_LANG = float(os.getenv("ISOCHRONY_MIN_SIM_CROSS_LANG", "0.60"))

# How much semantic fidelity counts relative to timing fit in the combined score.
DEFAULT_SEMANTIC_WEIGHT = float(os.getenv("ISOCHRONY_SEMANTIC_WEIGHT", "0.4"))

_DISABLED_WARNING_EMITTED = False


class SemanticGate:
    """Multilingual sentence embedder wrapper for meaning-preservation checks.

    Lazily loaded so importing this module costs nothing until a translation actually
    runs, and so a missing optional dependency degrades to "gate off" rather than an
    ImportError at module import time.
    """

    def __init__(self, model_id: str = DEFAULT_EMBEDDER_MODEL_ID,
                 device: Optional[str] = None, enabled: bool = True):
        self.model_id = model_id
        self.device = device
        self.enabled = enabled
        self._model = None
        self._load_failed = False

    def _lazy_load(self):
        global _DISABLED_WARNING_EMITTED
        if self._model is not None or self._load_failed or not self.enabled:
            return self._model
        try:
            from sentence_transformers import SentenceTransformer
            logger.info("Loading semantic fidelity embedder %s ...", self.model_id)
            self._model = SentenceTransformer(self.model_id, device=self.device)
        except Exception as e:  # noqa: BLE001
            self._load_failed = True
            if not _DISABLED_WARNING_EMITTED:
                _DISABLED_WARNING_EMITTED = True
                logger.warning(
                    "Semantic fidelity gate DISABLED (%s: %s). Candidate selection will "
                    "fall back to timing-only scoring, which cannot detect a translation "
                    "that hit its phoneme budget by dropping meaning. Install with: "
                    "pip install sentence-transformers",
                    type(e).__name__, e,
                )
        return self._model

    @property
    def available(self) -> bool:
        return self._lazy_load() is not None

    def similarities(self, anchor: str, candidates: Sequence[str]) -> Optional[list[float]]:
        """Cosine similarity of each candidate against the anchor.

        Returns None when the gate is unavailable, so callers can distinguish
        "not checked" from "checked and scored 0.0". One batched forward pass.
        """
        model = self._lazy_load()
        if model is None or not candidates:
            return None
        try:
            import numpy as np
            embeddings = model.encode(
                [anchor, *candidates], normalize_embeddings=True, show_progress_bar=False
            )
            anchor_vec = embeddings[0]
            return [float(np.dot(anchor_vec, v)) for v in embeddings[1:]]
        except Exception as e:  # noqa: BLE001
            logger.warning("Semantic similarity computation failed: %s", e, exc_info=True)
            return None

    def similarity(self, text_a: str, text_b: str) -> Optional[float]:
        result = self.similarities(text_a, [text_b])
        return result[0] if result else None


def combined_score(timing_score: float, semantic_similarity: Optional[float],
                   min_similarity: float,
                   semantic_weight: float = DEFAULT_SEMANTIC_WEIGHT) -> float:
    """Blend timing fit with meaning preservation.

    Timing-only scoring optimises for the shortest text that fits, which is the wrong
    objective — the goal is the *most faithful* text that fits, not the shortest.

    Candidates at or above `min_similarity` are ranked on a weighted blend. Candidates
    below it are pushed beneath every passing candidate but keep their relative order,
    so that if everything fails the floor we still select the least-bad option rather
    than an arbitrary one.

    When similarity is None (gate unavailable) this returns the timing score unchanged,
    preserving the previous behaviour exactly.
    """
    if semantic_similarity is None:
        return timing_score
    blended = (1.0 - semantic_weight) * timing_score + semantic_weight * semantic_similarity
    if semantic_similarity < min_similarity:
        return blended - 1.0
    return blended


In [ ]:
%%writefile /kaggle/working/pipeline_v3/evaluation/phoneme_adherence_eval.py
"""
evaluation/phoneme_adherence_eval.py
=====================================
Checkpoint-trajectory evaluation for the length-constrained translation fine-tune.

WHAT CHANGED IN THIS REVISION, AND WHY
---------------------------------------
The previous harness produced numbers that could not be trusted, in four separate ways.
Each fix below corresponds to a defect that was found in its output, not to a style
preference.

**1. chrF++ was being swallowed.** `import sacrebleu` sat inside the same `try` as the
scoring call, under a bare `except Exception: return None`. When the pip install failed in
a Kaggle session, every one of 440 calls returned None without a single log line, the run
completed, a report was written, and the fidelity column came out blank — the one column
that would have told us whether the length constraint was being paid for out of meaning.
The import now happens once at module scope and its failure is logged loudly and recorded
in the report header, so an absent metric is visible rather than merely empty.

**2. There were two different slope estimators and the report used the wrong one.**
They are not duplicates; they measure different things, and the distinction is the whole
point of the metric:

  - *population slope* regresses generated length against requested length across
    **different sentences**, whose budgets differ because the sentences differ. A model
    that ignores the budget entirely still scores high on it — longer English produces
    longer Hindi regardless. It measures whether translations are appropriately scaled,
    which is a fluency property, not budget obedience.
  - *probe slope* holds the sentence **fixed** and sweeps only the requested budget across
    0.6-1.4x. The sentence is constant, so the only thing that can move the output length
    is the budget. This is the capability probe.

The old report quoted the population slope while the surrounding prose described the
probe. Both are now computed, named distinctly, and the **probe** is what the report
leads with.

**3. The probe's budgets were derived from the corpus labels**, which are known to be
mislabelled (see `common/phonemes.py`). Natural length is now measured from the reference
text with the canonical counter, so the probe is independent of the corpus labels and
measures capability against the true ruler.

**4. `summarize` used the population variance divisor and returned the upper-middle value
as the "median"** for even-length samples. Small, systematic, and in every median in every
report. Now sample variance (n-1) and a true median.

Two things were also missing rather than broken:

**Semantic fidelity was never measured.** chrF++ needs a reference translation, which
exists here and never exists at dub time. The production-side question — "how much meaning
did compression cost, measured against something available at inference?" — is answered by
scoring each generation against the full-budget candidate using the same embedder the
inference gate uses. That produces a degraded-segment rate per language, which is the
number that converts an architectural worry into evidence.

**The stopping rule lived in prose.** It is now `stopping_verdict()`, computed from the
trajectory and printed in the report, so the decision cannot be re-argued after the fact.

READING THE OUTPUT
------------------
Read the **per-language** table first, then the aggregate. Eleven languages across two
families do not plateau together — Dravidian languages are agglutinative, so their
token-to-phoneme relationship differs and they learn this task on a different schedule.
Every aggregate number hides that. Building the decomposition and then reading the
aggregate column anyway is the mistake this harness was already capable of preventing.
"""

from __future__ import annotations

import argparse
import glob
import json
import logging
import math
import os
import re
import statistics
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.languages import LANGUAGES, get_language  # noqa: E402
from common.phonemes import (  # noqa: E402
    PhonemizationError, assert_g2p_available, count_phonemes, ruler_id,
)

logger = logging.getLogger("phoneme_adherence_eval")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

PROMPT_RE = re.compile(r"\[Target Phonemes:\s*(\d+)\]")
LENGTH_SWEEP_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]

# --- chrF++ availability, resolved once, loudly -----------------------------------------
# Imported at module scope precisely so its absence is a visible fact about the run rather
# than 440 silent Nones.
try:
    import sacrebleu as _sacrebleu
    CHRF_AVAILABLE = True
    CHRF_UNAVAILABLE_REASON = None
except Exception as _e:  # noqa: BLE001
    _sacrebleu = None
    CHRF_AVAILABLE = False
    CHRF_UNAVAILABLE_REASON = repr(_e)
    logger.error(
        "sacrebleu is NOT importable (%s). chrF++ will be absent from this report. "
        "Fidelity is the axis a length-targeted fine-tune puts at risk, so a run without "
        "it answers a strictly smaller question. Install with: pip install sacrebleu",
        CHRF_UNAVAILABLE_REASON,
    )


# ========================================================================================
# Corpus IO
# ========================================================================================

def read_val(path: str) -> list:
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        logger.warning("%d unparseable lines skipped in %s", bad, path)
    return rows


def group_by_language(rows: list) -> dict:
    g = defaultdict(list)
    for r in rows:
        g[r.get("language", "unknown")].append(r)
    return g


def requested_n(row: dict) -> int:
    """The budget the row's own prompt states — i.e. what the model was actually asked for.

    Read from the prompt text first, not from the `n_phonemes` field, because the prompt
    is what the model sees. If a relabelling ever updates one and not the other, this
    reports the number that actually conditioned the generation.
    """
    m = PROMPT_RE.search(row.get("prompt", ""))
    if m:
        return int(m.group(1))
    return int(row.get("n_phonemes") or 0)


def true_phoneme_len(text: str, lang: str) -> Optional[int]:
    try:
        return count_phonemes(text, lang)
    except PhonemizationError as e:
        logger.warning("phonemization failed (%s): %s", lang, e)
        return None


# ========================================================================================
# Statistics
# ========================================================================================

def summarize(xs: list) -> dict:
    """Sample statistics. Sample stdev (n-1) and a true median — the previous version used
    the population divisor and `xs_sorted[n // 2]`, which returns the upper middle value
    for even n."""
    if not xs:
        return {"n": 0}
    n = len(xs)
    return {
        "n": n,
        "mean": statistics.fmean(xs),
        "median": statistics.median(xs),
        "std": statistics.stdev(xs) if n > 1 else 0.0,
    }


def linfit_slope(xs: list, ys: list):
    n = len(xs)
    if n < 2:
        return None, None
    mx, my = sum(xs) / n, sum(ys) / n
    sxx = sum((x - mx) ** 2 for x in xs)
    sxy = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    if sxx == 0:
        return None, None
    slope = sxy / sxx
    syy = sum((y - my) ** 2 for y in ys)
    r2 = (sxy * sxy) / (sxx * syy) if syy > 0 else None
    return slope, r2


def chrf_pp(hypothesis: str, reference: str) -> Optional[float]:
    """chrF++ against the reference. Returns None only when sacrebleu is genuinely absent
    — which is recorded once, at module import, rather than per call."""
    if not CHRF_AVAILABLE:
        return None
    return float(_sacrebleu.sentence_chrf(hypothesis, [reference], word_order=2).score)


# ========================================================================================
# Semantic scoring — the production-side fidelity check
# ========================================================================================

class SemanticScorer:
    """Cosine similarity with the same embedder the inference gate uses.

    Deliberately the same model as `translation/semantic_gate.py` and
    `training/length_augmentation.py`: a threshold validated on one embedder means nothing
    on another, so an eval that used a different one could not be compared against the
    gate that ships.
    """

    def __init__(self, model_id: Optional[str] = None, device: Optional[str] = None):
        from translation.semantic_gate import DEFAULT_EMBEDDER_MODEL_ID
        self.model_id = model_id or DEFAULT_EMBEDDER_MODEL_ID
        self.device = device
        self._model = None
        self.available = True
        self.reason = None

    def _lazy(self):
        if self._model is None and self.available:
            try:
                from sentence_transformers import SentenceTransformer
                logger.info("Loading semantic embedder %s ...", self.model_id)
                self._model = SentenceTransformer(self.model_id, device=self.device)
            except Exception as e:  # noqa: BLE001
                self.available = False
                self.reason = repr(e)
                logger.error("Semantic scoring DISABLED — embedder failed to load: %s", e)
        return self._model

    def similarity(self, a: str, b: str) -> Optional[float]:
        model = self._lazy()
        if model is None or not a or not b:
            return None
        import numpy as np
        emb = model.encode([a, b], normalize_embeddings=True)
        return float(np.dot(emb[0], emb[1]))


# ========================================================================================
# Model loading and generation
# ========================================================================================

def load_model(adapter_path: Optional[str], base_model_id: str, max_seq_length: int = 512):
    from unsloth import FastLanguageModel
    src = adapter_path if adapter_path else base_model_id
    model, tok = FastLanguageModel.from_pretrained(
        src, max_seq_length=max_seq_length, load_in_4bit=True, dtype=None,
    )
    FastLanguageModel.for_inference(model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model, tok


def _clean(text: str) -> str:
    text = text.strip()
    for prefix in ("Translation:", "Output:", "Target:"):
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    if len(text) >= 2 and text[0] in "\"'" and text[-1] in "\"'":
        text = text[1:-1].strip()
    return text


def generate_batch(model, tok, prompts: list[str], max_new_tokens: int = 128,
                   temperature: float = 0.3, batch_size: int = 8) -> list[str]:
    """Batched generation.

    The probe needs 5 generations per sentence per language per checkpoint; unbatched that
    dominates the entire session's wall clock and is the reason the probe was previously
    run at 10 sentences per language, where per-language orderings are not trustworthy.
    Left padding is required — decoder-only models continue from the rightmost token, and
    right padding would have the model continue from pad tokens.
    """
    import torch
    outs: list[str] = []
    prev_side = tok.padding_side
    tok.padding_side = "left"
    try:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            texts = [
                tok.apply_chat_template([{"role": "user", "content": p}],
                                        tokenize=False, add_generation_prompt=True)
                for p in chunk
            ]
            enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                gen = model.generate(
                    **enc, max_new_tokens=max_new_tokens,
                    do_sample=temperature > 0, temperature=max(temperature, 1e-4),
                    top_p=0.9, pad_token_id=tok.pad_token_id,
                )
            for j in range(len(chunk)):
                new = gen[j][enc["input_ids"].shape[1]:]
                outs.append(_clean(tok.decode(new, skip_special_tokens=True)))
    finally:
        tok.padding_side = prev_side
    return outs


def completion_ce(model, tok, prompt: str, completion: str, max_seq_length: int = 512) -> Optional[float]:
    import torch
    ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": completion}],
        return_tensors="pt", add_generation_prompt=False,
    )
    prompt_ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], return_tensors="pt", add_generation_prompt=True,
    )
    if ids.shape[1] > max_seq_length:
        return None
    labels = ids.clone()
    labels[:, : prompt_ids.shape[1]] = -100
    ids, labels = ids.to(model.device), labels.to(model.device)
    with torch.no_grad():
        loss = model(input_ids=ids, labels=labels).loss
    return float(loss.item())


# ========================================================================================
# Metric passes
# ========================================================================================

def eval_checkpoint(model, tok, rows_by_lang: dict, adherence_per_lang: int,
                    ce_per_lang: int, do_generation: bool, do_ce: bool,
                    semantic: Optional[SemanticScorer] = None,
                    semantic_threshold: float = 0.80,
                    budget_scale: Optional[dict] = None,
                    batch_size: int = 8,
                    dump_langs: Optional[set] = None, dump_n: int = 0,
                    dump_sink: Optional[list] = None, ckpt_label: str = "") -> dict:
    results = {}
    dump_langs = dump_langs or set()

    for lang, rows in rows_by_lang.items():
        if lang not in LANGUAGES:
            continue
        lang_res = {"language": lang}

        if do_ce:
            ce_vals = []
            for r in rows[:ce_per_lang]:
                c = completion_ce(model, tok, r["prompt"], r.get("completion") or r["target"])
                if c is not None:
                    ce_vals.append(c)
            s = summarize(ce_vals)
            lang_res["ce_mean"] = s.get("mean")
            lang_res["ce_perplexity"] = math.exp(s["mean"]) if s.get("mean") is not None else None
            lang_res["ce_n"] = s["n"]

        if do_generation:
            subset = [r for r in rows[:adherence_per_lang] if requested_n(r) > 0]
            prompts = [_rescaled_prompt(r, lang, budget_scale) for r in subset]
            gens = generate_batch(model, tok, prompts, temperature=0.3, batch_size=batch_size)

            rel_errs, signed_errs, chrfs, sims = [], [], [], []
            req_ns, gen_ns = [], []
            degraded = 0
            dumped = 0
            for r, gen in zip(subset, gens):
                if not gen:
                    continue
                # The budget we hold the model to is always in TRUE phonemes, whatever unit
                # the corpus label happened to be written in.
                N = _true_budget(r, lang)
                if not N:
                    continue
                n_gen = true_phoneme_len(gen, lang)
                if n_gen is None:
                    continue
                ref = r.get("target") or r.get("completion") or ""
                rel_errs.append(abs(n_gen - N) / N)
                signed_errs.append((n_gen - N) / N)
                req_ns.append(float(N))
                gen_ns.append(float(n_gen))

                c = chrf_pp(gen, ref) if ref else None
                if c is not None:
                    chrfs.append(c)

                sim = None
                if semantic is not None:
                    # Anchor = the human reference. At dub time no reference exists and the
                    # anchor is the full-budget candidate instead (see probe below); here the
                    # reference is the stronger anchor and is free.
                    sim = semantic.similarity(ref, gen) if ref else None
                    if sim is not None:
                        sims.append(sim)
                        if sim < semantic_threshold:
                            degraded += 1

                if lang in dump_langs and dump_sink is not None and dumped < dump_n:
                    dump_sink.append({
                        "checkpoint": ckpt_label, "language": lang, "english": r.get("english"),
                        "requested_n": N, "generated_n": n_gen, "generated": gen,
                        "reference": ref, "chrf": c, "semantic_similarity": sim,
                    })
                    dumped += 1

            ra, sa, ca, si = summarize(rel_errs), summarize(signed_errs), summarize(chrfs), summarize(sims)
            lang_res["adherence_rel_mean"] = ra.get("mean")
            lang_res["adherence_rel_median"] = ra.get("median")
            lang_res["adherence_signed_mean"] = sa.get("mean")
            lang_res["adherence_signed_median"] = sa.get("median")
            lang_res["chrf_mean"] = ca.get("mean")
            lang_res["semantic_mean"] = si.get("mean")
            lang_res["semantic_degraded_frac"] = (degraded / si["n"]) if si.get("n") else None
            lang_res["adherence_n"] = ra["n"]
            slope, r2 = linfit_slope(req_ns, gen_ns)
            # Named for what it is. Across DIFFERENT sentences, so it is confounded by
            # sentence length and is a fluency proxy, not a budget-obedience measurement.
            lang_res["length_slope_population"] = slope
            lang_res["length_r2_population"] = r2

        results[lang] = lang_res
    return results


def _true_budget(row: dict, lang: str) -> Optional[int]:
    """The budget in true phonemes.

    If the row was written by the repaired labeller it carries `ruler`, and `n_phonemes`
    is already correct. Otherwise the reference text is re-counted, so a legacy
    character-ruled corpus is still evaluated on the right ruler.
    """
    if str(row.get("ruler", "")).startswith("phonemes:"):
        n = int(row.get("n_phonemes") or 0)
        if n > 0:
            return n
    ref = row.get("target") or row.get("completion") or ""
    return true_phoneme_len(ref, lang) if ref else None


def _rescaled_prompt(row: dict, lang: str, budget_scale: Optional[dict]) -> str:
    """The prompt to send.

    With `--budget_scale_json`, the true-phoneme budget is divided by that language's
    phonemes-per-character constant before being written into the prompt. That converts
    "the budget I want, in phonemes" into "the budget this model was actually taught, in
    characters" — the salvage path for a checkpoint trained on character-ruled labels,
    which needs no retraining. Without the flag the row's own prompt is used unchanged.
    """
    if not budget_scale or lang not in budget_scale:
        return row["prompt"]
    N = _true_budget(row, lang)
    if not N:
        return row["prompt"]
    k = budget_scale[lang]
    asked = max(1, round(N / k)) if k else N
    return PROMPT_RE.sub(f"[Target Phonemes: {asked}]", row["prompt"])


def length_response_probe(model, tok, rows_by_lang: dict, sentences_per_lang: int,
                          semantic: Optional[SemanticScorer] = None,
                          semantic_threshold: float = 0.80,
                          budget_scale: Optional[dict] = None,
                          batch_size: int = 8) -> dict:
    """The capability probe: one sentence, five budgets, only the budget varies.

    Natural length is measured from the reference text with the canonical counter rather
    than read from the corpus label, so the probe is unaffected by how the corpus was
    labelled.

    The semantic anchor here is the model's own 1.0x generation — the same anchor the
    inference gate uses, because at dub time no reference exists. That makes the degraded
    rate reported here directly comparable to what the shipped gate will see.
    """
    out = {}
    for lang, rows in rows_by_lang.items():
        if lang not in LANGUAGES:
            continue
        lang_name = get_language(lang).name

        specs = []          # (sentence_index, factor, prompt)
        naturals = []
        for si, r in enumerate(rows[:sentences_per_lang]):
            eng = r.get("english") or ""
            ref = r.get("target") or r.get("completion") or ""
            if not eng or not ref:
                continue
            natural = true_phoneme_len(ref, lang)
            if not natural:
                continue
            naturals.append(natural)
            k = (budget_scale or {}).get(lang)
            for f in LENGTH_SWEEP_FACTORS:
                want = max(1, round(natural * f))          # what we want, in phonemes
                asked = max(1, round(want / k)) if k else want   # what we write in the prompt
                specs.append((len(naturals) - 1, f, want,
                              f'[Translate to {lang_name}] [Target Phonemes: {asked}] "{eng}"'))

        if not specs:
            out[lang] = {"language": lang, "length_slope_probe": None, "n_points": 0}
            continue

        gens = generate_batch(model, tok, [s[3] for s in specs], temperature=0.3,
                              batch_size=batch_size)

        req_all, gen_all = [], []
        by_sentence: dict[int, dict[float, str]] = defaultdict(dict)
        for (si, f, want, _), gen in zip(specs, gens):
            if not gen:
                continue
            n_gen = true_phoneme_len(gen, lang)
            if n_gen is None:
                continue
            req_all.append(float(want))
            gen_all.append(float(n_gen))
            by_sentence[si][f] = gen

        slope, r2 = linfit_slope(req_all, gen_all)

        # Semantic cost of compression, anchored the way production anchors it.
        sims, degraded, n_scored = [], 0, 0
        if semantic is not None:
            for si, byf in by_sentence.items():
                anchor = byf.get(1.0)
                if not anchor:
                    continue
                for f, gen in byf.items():
                    if f >= 1.0:
                        continue
                    s = semantic.similarity(anchor, gen)
                    if s is None:
                        continue
                    sims.append(s)
                    n_scored += 1
                    if s < semantic_threshold:
                        degraded += 1

        out[lang] = {
            "language": lang,
            "length_slope_probe": slope,
            "length_r2_probe": r2,
            "n_points": len(req_all),
            "n_sentences": len(naturals),
            "compressed_semantic_mean": statistics.fmean(sims) if sims else None,
            "compressed_degraded_frac": (degraded / n_scored) if n_scored else None,
        }
    return out


# ========================================================================================
# The stopping rule, as code
# ========================================================================================

def stopping_verdict(traj: list[dict], ce_flat_tol: float = 0.005,
                     slope_move_tol: float = 0.01) -> dict:
    """CE-flat + slope still moving => keep spending quota.
       CE-flat + slope flat        => genuine plateau; early-stop loses nothing.

    Written as code rather than kept in prose because the whole point of the rule is that
    it must survive the moment when stopping looks attractive. Uses the probe slope; the
    population slope is not a capability measurement.
    """
    pts = sorted([t for t in traj if t.get("step", -1) >= 0], key=lambda x: x["step"])
    if len(pts) < 2:
        return {"verdict": "INSUFFICIENT_DATA", "reason": "need at least two checkpoints"}

    def last_valid(key):
        vals = [(t["step"], t[key]) for t in pts if t.get(key) is not None]
        return vals[-2:] if len(vals) >= 2 else None

    ce = last_valid("ce_mean")
    sl = last_valid("length_slope_probe") or last_valid("length_slope_population")
    if sl is None:
        return {"verdict": "INSUFFICIENT_DATA", "reason": "no slope measured on >=2 checkpoints"}

    ce_delta = (ce[1][1] - ce[0][1]) if ce else None
    slope_delta = sl[1][1] - sl[0][1]
    ce_flat = ce_delta is None or abs(ce_delta) < ce_flat_tol
    slope_moving = slope_delta > slope_move_tol

    if ce_flat and slope_moving:
        verdict, reason = "CONTINUE", (
            f"CE flat (delta {ce_delta:+.4f}) but probe slope still climbing "
            f"({sl[0][1]:.3f} -> {sl[1][1]:.3f}, delta {slope_delta:+.3f}). Length control "
            f"is still being learned after the loss curve went quiet. Keep spending quota."
        )
    elif ce_flat:
        verdict, reason = "STOP", (
            f"CE flat (delta {ce_delta if ce_delta is None else f'{ce_delta:+.4f}'}) and probe "
            f"slope flat ({sl[0][1]:.3f} -> {sl[1][1]:.3f}, delta {slope_delta:+.3f}). "
            f"Genuine plateau; early-stopping loses nothing measurable."
        )
    else:
        verdict, reason = "CONTINUE", (
            f"CE still moving (delta {ce_delta:+.4f}). Not a plateau."
        )
    return {"verdict": verdict, "reason": reason,
            "ce_delta": ce_delta, "slope_delta": slope_delta,
            "steps_compared": [sl[0][0], sl[1][0]]}


# ========================================================================================
# Driver
# ========================================================================================

def checkpoint_step(path: str) -> int:
    m = re.search(r"checkpoint-(\d+)", os.path.basename(str(path).rstrip("/")))
    return int(m.group(1)) if m else -1


def run(args):
    # An eval that silently used a character fallback would report a mislabelled corpus as
    # correctly labelled. Refuse to start.
    g2p = assert_g2p_available()
    logger.info("G2P ruler: %s", g2p["ruler"])

    budget_scale = None
    if args.budget_scale_json:
        budget_scale = json.loads(Path(args.budget_scale_json).read_text(encoding="utf-8"))
        if "per_language" in budget_scale:  # accept a ruler_audit report directly
            budget_scale = {k: v["ols_k_through_origin"]
                            for k, v in budget_scale["per_language"].items()}
        logger.info("Budget rescale ACTIVE: %s", budget_scale)

    rows = read_val(args.val_jsonl)
    rows_by_lang = group_by_language(rows)
    logger.info("Loaded %d val rows across %d languages", len(rows), len(rows_by_lang))

    semantic = None
    if not args.no_semantic:
        semantic = SemanticScorer(device=args.semantic_device)

    targets = []
    if args.base_baseline:
        targets.append(("base_model", None))
    ckpts = list(args.checkpoints or [])
    if args.checkpoints_glob:
        found = glob.glob(args.checkpoints_glob)
        if not found:
            # The silent-resume bug's twin: a glob that matches nothing produces a run that
            # looks healthy and evaluates nothing.
            raise SystemExit(
                f"--checkpoints_glob {args.checkpoints_glob!r} matched NOTHING. "
                f"Check the directory depth before spending a session on it."
            )
        ckpts += found
    ckpts = sorted(set(ckpts), key=checkpoint_step)
    for c in ckpts:
        targets.append((os.path.basename(str(c).rstrip("/")), c))
    if not targets:
        raise SystemExit("Nothing to evaluate: pass --checkpoints/--checkpoints_glob and/or --base_baseline")

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    do_gen = args.mode in ("all", "adherence", "length")
    do_ce = args.mode in ("all", "ce")
    do_probe = args.mode in ("all", "length")

    per_ckpt_rows, traj_rows, lr_rows, sample_rows = [], [], [], []
    dump_langs = set(args.dump_samples_langs or [])

    wb = wandb_init({"ruler": g2p["ruler"], "val_jsonl": args.val_jsonl, "mode": args.mode,
                     "checkpoints": [t[0] for t in targets],
                     "budget_scale": budget_scale}, args.wandb_project,
                    args.wandb_entity, args.wandb_run_name) if args.wandb else None

    for label, adapter in targets:
        logger.info("=== Evaluating %s ===", label)
        model, tok = load_model(adapter, args.base_model_id, args.max_seq_length)
        res = eval_checkpoint(
            model, tok, rows_by_lang, args.adherence_samples_per_lang,
            args.ce_samples_per_lang, do_gen, do_ce,
            semantic=semantic, semantic_threshold=args.semantic_threshold,
            budget_scale=budget_scale, batch_size=args.batch_size,
            dump_langs=dump_langs, dump_n=args.dump_samples_n,
            dump_sink=sample_rows, ckpt_label=label,
        )
        probe = length_response_probe(
            model, tok, rows_by_lang, args.probe_sentences_per_lang,
            semantic=semantic, semantic_threshold=args.semantic_threshold,
            budget_scale=budget_scale, batch_size=args.batch_size,
        ) if do_probe else {}

        def agg(source: dict, key: str):
            vals = [v[key] for v in source.values() if v.get(key) is not None]
            return sum(vals) / len(vals) if vals else None

        step = checkpoint_step(adapter) if adapter else -1
        traj_rows.append({
            "checkpoint": label, "step": step,
            "ce_mean": agg(res, "ce_mean"), "ce_perplexity": agg(res, "ce_perplexity"),
            "adherence_rel_mean": agg(res, "adherence_rel_mean"),
            "adherence_signed_mean": agg(res, "adherence_signed_mean"),
            "chrf_mean": agg(res, "chrf_mean"),
            "semantic_mean": agg(res, "semantic_mean"),
            "semantic_degraded_frac": agg(res, "semantic_degraded_frac"),
            "length_slope_population": agg(res, "length_slope_population"),
            "length_slope_probe": agg(probe, "length_slope_probe") if probe else None,
            "compressed_degraded_frac": agg(probe, "compressed_degraded_frac") if probe else None,
        })
        for lang, v in res.items():
            merged = {**v, **{k: val for k, val in (probe.get(lang) or {}).items()
                              if k != "language"}}
            per_ckpt_rows.append({"checkpoint": label, "step": step, **merged})
        for lang, v in probe.items():
            lr_rows.append({"checkpoint": label, "step": step, **v})

        _write_csv(out_dir / "per_checkpoint_metrics.csv", per_ckpt_rows)
        _write_csv(out_dir / "trajectory_summary.csv", traj_rows)
        if lr_rows:
            _write_csv(out_dir / "length_response.csv", lr_rows)
        # Stream to W&B on the same cadence as the CSVs — this is the only signal
        # visible while the session is still running.
        wandb_log_checkpoint(wb, traj_rows[-1],
                             [r for r in per_ckpt_rows if r["step"] == step])

        del model
        try:
            import gc
            import torch
            gc.collect(); torch.cuda.empty_cache()
        except Exception:  # noqa: BLE001
            pass

    if sample_rows:
        with open(out_dir / "samples.jsonl", "w", encoding="utf-8") as f:
            for row in sample_rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    manifest = {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "ruler": g2p["ruler"],
        "val_jsonl": args.val_jsonl,
        "n_val_rows": len(rows),
        "mode": args.mode,
        "adherence_samples_per_lang": args.adherence_samples_per_lang,
        "probe_sentences_per_lang": args.probe_sentences_per_lang,
        "chrf_available": CHRF_AVAILABLE,
        "chrf_unavailable_reason": CHRF_UNAVAILABLE_REASON,
        "semantic_available": bool(semantic and semantic.available),
        "semantic_threshold": args.semantic_threshold,
        "budget_scale": budget_scale,
        "checkpoints": [t[0] for t in targets],
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    _write_report(out_dir / "eval_report.md", traj_rows, per_ckpt_rows, lr_rows, manifest)
    if wb is not None:
        try:
            import wandb as _wb
            cols = ["checkpoint", "step", "language", "adherence_rel_mean",
                    "adherence_signed_mean", "length_slope_probe",
                    "length_slope_population", "chrf_mean", "semantic_mean",
                    "semantic_degraded_frac", "ce_mean"]
            tbl = _wb.Table(columns=cols)
            for r in per_ckpt_rows:
                tbl.add_data(*[r.get(c) for c in cols])
            wb.log({"per_language": tbl})
            v = stopping_verdict(traj_rows)
            wb.summary["stopping_verdict"] = v["verdict"]
            wb.summary["stopping_reason"] = v.get("reason", "")
            wb.summary["ruler"] = manifest.get("ruler")
            wb.finish()
            logger.info("wandb run CLOSED")
        except Exception as e:  # noqa: BLE001
            logger.error("wandb finalisation failed: %s", e)
    logger.info("Wrote outputs to %s", out_dir)


def wandb_init(manifest: dict, project: str, entity: Optional[str],
               run_name: Optional[str]):
    """Opens the W&B run BEFORE evaluation starts, so metrics can stream.

    The first version of this logged everything in one call at the end of `run()`. That
    makes W&B useless for its actual job here: Kaggle publishes a notebook's log only when
    the session ends, so W&B is the only live signal during a multi-hour run — and a
    channel that reports nothing until the run is over is not a live signal. Metrics are
    now logged after each checkpoint completes, matching the CSV writes.
    """
    try:
        import wandb
    except ImportError:
        logger.warning("wandb not installed — skipping (pip install wandb)")
        return None
    try:
        run = wandb.init(project=project, entity=entity, name=run_name,
                         job_type="evaluation", config=manifest, reinit=True)
        logger.info("wandb run OPEN: %s", getattr(run, "url", ""))
        return run
    except Exception as e:  # noqa: BLE001
        logger.error("wandb.init failed (%s) — continuing without it. The evaluation "
                     "artifacts on disk are unaffected.", e)
        return None


def wandb_log_checkpoint(run, traj_row: dict, lang_rows: list[dict]) -> None:
    """Streams one checkpoint's metrics as soon as it finishes."""
    if run is None:
        return
    step = traj_row["step"] if traj_row["step"] >= 0 else 0
    payload = {f"agg/{k}": v for k, v in traj_row.items()
               if k not in ("checkpoint", "step") and v is not None}
    for r in lang_rows:
        lang = r.get("language")
        for k in ("adherence_rel_mean", "adherence_signed_mean", "length_slope_probe",
                  "length_slope_population", "chrf_mean", "semantic_mean",
                  "semantic_degraded_frac", "ce_mean"):
            if r.get(k) is not None:
                payload[f"{lang}/{k}"] = r[k]
    try:
        run.log(payload, step=step)
        logger.info("wandb: logged %d metrics at step %d", len(payload), step)
    except Exception as e:  # noqa: BLE001
        logger.error("wandb log failed at step %s: %s", step, e)


def log_to_wandb(traj: list[dict], per_ckpt: list[dict], manifest: dict,
                 project: str, entity: Optional[str], run_name: Optional[str]) -> None:
    """Mirrors the report into Weights & Biases.

    What gets logged is deliberately not what was logged last time. CE and perplexity go up
    as *diagnostics*; the panels that matter are `length_slope_probe`, signed adherence, and
    the semantic degraded rate — the metrics that measure the objective. Logging CE
    prominently is how a run gets stopped on the wrong signal, and this project has already
    paid for that once.

    Per-language series are logged individually (`as/length_slope_probe`, …) because the
    aggregate hides that eleven languages across two families do not plateau together.
    """
    try:
        import wandb
    except ImportError:
        logger.warning("wandb not installed — skipping (pip install wandb)")
        return

    try:
        run = wandb.init(project=project, entity=entity, name=run_name,
                         job_type="evaluation", config=manifest, reinit=True)
    except Exception as e:  # noqa: BLE001
        logger.error("wandb.init failed (%s) — continuing without it. The evaluation "
                     "artifacts on disk are unaffected.", e)
        return

    for t in sorted(traj, key=lambda x: x["step"]):
        step = t["step"] if t["step"] >= 0 else 0
        payload = {f"agg/{k}": v for k, v in t.items()
                   if k not in ("checkpoint", "step") and v is not None}
        for r in per_ckpt:
            if r["step"] != t["step"]:
                continue
            lang = r.get("language")
            for k in ("adherence_rel_mean", "adherence_signed_mean", "length_slope_probe",
                      "length_slope_population", "chrf_mean", "semantic_mean",
                      "semantic_degraded_frac", "ce_mean"):
                if r.get(k) is not None:
                    payload[f"{lang}/{k}"] = r[k]
        run.log(payload, step=step)

    cols = ["checkpoint", "step", "language", "adherence_rel_mean", "adherence_signed_mean",
            "length_slope_probe", "length_slope_population", "chrf_mean", "semantic_mean",
            "semantic_degraded_frac", "ce_mean"]
    tbl = wandb.Table(columns=cols)
    for r in per_ckpt:
        tbl.add_data(*[r.get(c) for c in cols])
    run.log({"per_language": tbl})

    v = stopping_verdict(traj)
    run.summary["stopping_verdict"] = v["verdict"]
    run.summary["stopping_reason"] = v.get("reason", "")
    run.summary["ruler"] = manifest.get("ruler")
    run.finish()
    logger.info("wandb run: %s", getattr(run, "url", ""))


def _write_csv(path, rows):
    if not rows:
        return
    cols = list({k for r in rows for k in r.keys()})
    order = ["checkpoint", "step", "language"]
    cols = [c for c in order if c in cols] + sorted(c for c in cols if c not in order)
    # csv.writer rather than manual joining: any value containing a comma — a ruler string,
    # a language name, a failure message — silently shifts every subsequent column when you
    # join by hand, and the file still parses, just wrongly.
    import csv
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for r in rows:
            w.writerow(["" if r.get(c) is None else r.get(c) for c in cols])


def _fmt(x, p=3):
    return "-" if x is None else f"{x:.{p}f}"


def _write_report(path, traj, per_ckpt, lr, manifest):
    L = ["# Phoneme-Adherence Evaluation Report", ""]
    L += [f"- **Ruler:** `{manifest['ruler']}`",
          f"- **Val set:** `{manifest['val_jsonl']}` ({manifest['n_val_rows']} rows)",
          f"- **Generated:** {manifest['generated_utc']}",
          f"- **Adherence samples/lang:** {manifest['adherence_samples_per_lang']}  "
          f"**Probe sentences/lang:** {manifest['probe_sentences_per_lang']}"]
    if manifest.get("budget_scale"):
        L.append(f"- **Budget rescale ACTIVE** (phoneme budget converted to the character "
                 f"budget the model was taught): `{manifest['budget_scale']}`")
    if not manifest["chrf_available"]:
        L.append(f"- **chrF++ UNAVAILABLE** — `{manifest['chrf_unavailable_reason']}`. "
                 f"The fidelity column is absent, not zero.")
    if not manifest["semantic_available"]:
        L.append("- **Semantic scoring UNAVAILABLE** — embedder failed to load. "
                 "Semantic columns are absent, not zero.")
    L += ["", "> Read the per-language table first. Eleven languages across two families do "
          "not plateau together; every aggregate number below hides that.", ""]

    # --- per-language first, by design ---
    L += ["## Per-language", ""]
    by_ckpt = defaultdict(list)
    for r in per_ckpt:
        by_ckpt[(r["step"], r["checkpoint"])].append(r)
    for (step, ckpt) in sorted(by_ckpt):
        L += [f"### {ckpt} (step {step})", "",
              "| Lang | relErr | signed | probe slope | pop slope | chrF++ | semantic | degraded |",
              "|---|---|---|---|---|---|---|---|"]
        for r in sorted(by_ckpt[(step, ckpt)], key=lambda x: x.get("language") or ""):
            L.append(
                f"| {r.get('language')} | {_fmt(r.get('adherence_rel_mean'))} | "
                f"{_fmt(r.get('adherence_signed_mean'))} | {_fmt(r.get('length_slope_probe'))} | "
                f"{_fmt(r.get('length_slope_population'))} | {_fmt(r.get('chrf_mean'), 1)} | "
                f"{_fmt(r.get('semantic_mean'))} | {_fmt(r.get('semantic_degraded_frac'))} |")
        L.append("")

    # --- aggregate ---
    L += ["## Aggregate (read second)", "",
          "| Checkpoint | Step | CE | PPL | relErr | signed | chrF++ | semantic | probe slope | pop slope |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for t in sorted(traj, key=lambda x: x["step"]):
        L.append(f"| {t['checkpoint']} | {t['step']} | {_fmt(t['ce_mean'], 4)} | "
                 f"{_fmt(t['ce_perplexity'])} | {_fmt(t['adherence_rel_mean'])} | "
                 f"{_fmt(t['adherence_signed_mean'])} | {_fmt(t['chrf_mean'], 1)} | "
                 f"{_fmt(t.get('semantic_mean'))} | {_fmt(t.get('length_slope_probe'))} | "
                 f"{_fmt(t['length_slope_population'])} |")

    v = stopping_verdict(traj)
    L += ["", "## Stopping verdict", "", f"**{v['verdict']}** — {v.get('reason', '')}", "",
          "> `length_slope_probe` holds the sentence fixed and sweeps only the budget: it is "
          "the capability measurement. `length_slope_population` regresses across different "
          "sentences and is confounded by sentence length — a model that ignores the budget "
          "entirely still scores high on it. Never select a checkpoint on the population slope.", ""]

    Path(path).write_text("\n".join(L) + "\n", encoding="utf-8")


def _cli():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--val_jsonl", required=True)
    p.add_argument("--checkpoints", nargs="*", default=[])
    p.add_argument("--checkpoints_glob", default=None)
    p.add_argument("--base_baseline", action="store_true")
    p.add_argument("--base_model_id", default="meta-llama/Llama-3.1-8B-Instruct")
    p.add_argument("--output_dir", required=True)
    p.add_argument("--mode", choices=["all", "ce", "adherence", "length"], default="all")
    p.add_argument("--max_seq_length", type=int, default=512)
    p.add_argument("--adherence_samples_per_lang", type=int, default=40)
    p.add_argument("--ce_samples_per_lang", type=int, default=150)
    p.add_argument("--probe_sentences_per_lang", type=int, default=30,
                   help="Sentences per language for the capability probe; each costs 5 "
                        "generations. The previous default of 10 gave 50 points per "
                        "language, at which per-language orderings are not trustworthy.")
    p.add_argument("--batch_size", type=int, default=8,
                   help="Generation batch size. The probe is generation-bound; batching is "
                        "what makes a trustworthy sentence count affordable.")
    p.add_argument("--budget_scale_json", default=None,
                   help="Path to a tools/ruler_audit.py report (or a plain {lang: k} map). "
                        "Converts the true-phoneme budget into the character budget a "
                        "character-ruled checkpoint was actually taught — the salvage path.")
    p.add_argument("--semantic_threshold", type=float, default=0.80)
    p.add_argument("--semantic_device", default=None)
    p.add_argument("--no_semantic", action="store_true",
                   help="Skip semantic scoring (faster; loses the production-side fidelity axis).")
    p.add_argument("--wandb", action="store_true", help="Mirror the report into W&B.")
    p.add_argument("--wandb_project", default="indic-dubbing-v3")
    # The personal namespace `nktthegreat` holds ZERO projects; everything lives
    # under the team entity. Getting this wrong sends metrics to a namespace nobody
    # looks at, and the run appears to have logged nothing.
    p.add_argument("--wandb_entity", default="nktthegreat-soccernet")
    p.add_argument("--wandb_run_name", default=None)
    p.add_argument("--dump_samples_langs", nargs="*", default=[])
    p.add_argument("--dump_samples_n", type=int, default=8)
    run(p.parse_args())


if __name__ == "__main__":
    _cli()


## 3. G2P preflight

The eval refuses to start without it — an evaluation that silently fell back to character counts would report a mislabelled corpus as correctly labelled, which is how this whole situation arose.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
from common.phonemes import assert_g2p_available, ruler_id
assert_g2p_available()
print("\nRULER:", ruler_id())


## 4. Locate inputs

Found and reported, not hard-coded.

In [ ]:
from pathlib import Path
import json, shutil

VAL = next(iter(Path("/kaggle/input").rglob("val.phonemes.jsonl")), None)
SCALE = next(iter(Path("/kaggle/input").rglob("budget_scale.json")), None)
print("VAL  :", VAL)
print("SCALE:", SCALE)
assert VAL, "attach the 02h-ruler-audit output"

rows = [json.loads(l) for l in open(VAL, encoding="utf-8") if l.strip()]
rulers = {r.get("ruler") for r in rows}
print(f"{len(rows)} val rows | rulers: {rulers}")
assert all(str(r).startswith("phonemes:") for r in rulers), "val set is not phoneme-ruled"

# Copy the adapters we want locally: PEFT wants a writable dir, and these are ~80 MB each.
WANT = [2558, 3200, 3400, 3801]
src_root = next(iter(Path("/kaggle/input").rglob("checkpoints/translation_llm")), None)
assert src_root, "attach the 02-llm-finetune output"
CKPTS = []
for s in WANT:
    src = src_root / f"checkpoint-{s}"
    if not src.exists():
        print("  MISSING", src); continue
    dst = Path(f"/kaggle/working/ckpt/checkpoint-{s}")
    if not dst.exists():
        shutil.copytree(src, dst)
    CKPTS.append(str(dst))
print("checkpoints staged:", CKPTS)


## 5. Pass A — the corrected baseline

Prompted with **true phoneme budgets**, scored in **true phonemes**. Both sides on the same
ruler for the first time.

`--probe_sentences_per_lang 25` gives 125 points per language. The old default of 10 gave
50, at which the per-language orderings the previous report relied on (te 0.282 vs ml 0.374)
were not trustworthy.

## 4b. End-to-end smoke test — ONE example, before committing hours

Four attempts at this notebook have died: a missing package, a P100 assignment, a
`--no-deps` version conflict, and an ordering bug. Each cost GPU quota and produced no
data. So before the 3-6 hour loop starts, exercise the entire path exactly once — load a
checkpoint, generate one translation, count its phonemes, score it semantically. If the
pipeline is broken, this says so in about four minutes.

This is the same discipline as overfitting twenty examples before a long training run, and
for the same reason: the expensive thing should never be the thing that discovers the
cheap bug.

In [ ]:
import time
from evaluation.phoneme_adherence_eval import (
    load_model, generate_batch, true_phoneme_len, chrf_pp, CHRF_AVAILABLE, SemanticScorer)

t0 = time.time()
row = next(r for r in rows if r.get("language") == "hi")
m, tk = load_model(CKPTS[-1], "meta-llama/Llama-3.1-8B-Instruct", 512)
gen = generate_batch(m, tk, [row["prompt"]], batch_size=1)[0]

n_got = true_phoneme_len(gen, "hi")
n_req = int(row["n_phonemes"])
print(f"prompt   : {row['prompt'][:110]}")
print(f"generated: {gen[:110]}")
print(f"requested {n_req} phonemes, produced {n_got}  (rel err {abs(n_got-n_req)/n_req:.3f})")
print(f"chrF++   : {chrf_pp(gen, row['target']) if CHRF_AVAILABLE else 'UNAVAILABLE'}")

sem = SemanticScorer()
print(f"semantic : {sem.similarity(row['target'], gen)}")

assert gen, "model produced empty output"
assert n_got, "generation could not be phonemized"
print(f"SMOKE PASSED in {time.time()-t0:.0f}s - the full path works. Proceeding.")

del m
import gc, torch
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Export before the shell cell uses them, not after.
import os
os.environ["VAL_PATH"] = str(VAL)
os.environ["CKPT_ARGS"] = " ".join(CKPTS)
print(os.environ["VAL_PATH"]); print(os.environ["CKPT_ARGS"])


In [ ]:
import subprocess, sys

# subprocess with check=True, NOT `!python ...`. A `!` cell's non-zero exit does not
# propagate in papermill, so a failed eval lets the notebook continue and the real error
# surfaces cells later as something unrelated. That is exactly how this notebook failed the
# first time (missing `unsloth` reported as a missing report file).
cmd = [sys.executable, "-m", "evaluation.phoneme_adherence_eval",
       "--val_jsonl", str(VAL),
       "--checkpoints", *CKPTS,
       "--base_baseline",
       "--output_dir", "/kaggle/working/eval_corrected",
       "--mode", "all",
       "--adherence_samples_per_lang", "30",
       "--ce_samples_per_lang", "60",
       "--probe_sentences_per_lang", "25",
       "--batch_size", "8",
       "--dump_samples_langs", "as", "or", "hi", "ta",
       "--dump_samples_n", "6",
       "--wandb", "--wandb_required", "--wandb_project", "indic-dubbing-v3",
       "--wandb_entity", "nktthegreat-soccernet",
       "--wandb_run_name", "02i-corrected-eval"]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)


## 6. Read the per-language table FIRST

Eleven languages across two families do not plateau together. The failure this project already made once was building the per-language decomposition and then reading the aggregate column anyway.

In [ ]:
from pathlib import Path
print(Path("/kaggle/working/eval_corrected/eval_report.md").read_text(encoding="utf-8"))


## 7. Test the prediction

Assamese and Odia were under-labelled by +16.5% and +15.2%, so the model should
**under-produce** in exactly those two. If they are not the most negative on signed
adherence, the causal story behind the RETRAIN decision is wrong.

In [ ]:
import csv
from pathlib import Path

rows = list(csv.DictReader(open("/kaggle/working/eval_corrected/per_checkpoint_metrics.csv", encoding="utf-8")))
last = max(int(r["step"]) for r in rows)
sel = [r for r in rows if int(r["step"]) == last]

def f(r, k):
    try: return float(r[k])
    except Exception: return None

sel.sort(key=lambda r: (f(r, "adherence_signed_mean") if f(r, "adherence_signed_mean") is not None else 9))
print(f"signed adherence at step {last}, most negative first:\n")
print(f"{'lang':<6}{'signed':>9}{'relErr':>9}{'probe':>9}{'chrF++':>9}{'sem':>8}")
for r in sel:
    print(f"{r['language']:<6}{f(r,'adherence_signed_mean') or 0:>9.3f}{f(r,'adherence_rel_mean') or 0:>9.3f}"
          f"{f(r,'length_slope_probe') or 0:>9.3f}{f(r,'chrf_mean') or 0:>9.1f}{f(r,'semantic_mean') or 0:>8.3f}")

worst2 = {r["language"] for r in sel[:2]}
print(f"\ntwo most-negative: {worst2}")
print("PREDICTION HELD" if worst2 & {"as", "or"} else
      "PREDICTION FAILED — revisit the causal story before retraining")


## 8. Stopping verdict

Computed from the trajectory, on the probe slope, as code.

In [ ]:
import csv, json
from evaluation.phoneme_adherence_eval import stopping_verdict

traj = []
for r in csv.DictReader(open("/kaggle/working/eval_corrected/trajectory_summary.csv", encoding="utf-8")):
    def g(k):
        try: return float(r[k])
        except Exception: return None
    traj.append({"checkpoint": r["checkpoint"], "step": int(r["step"]),
                 "ce_mean": g("ce_mean"), "length_slope_probe": g("length_slope_probe"),
                 "length_slope_population": g("length_slope_population")})
v = stopping_verdict(traj)
print(json.dumps(v, indent=2))
Path("/kaggle/working/eval_corrected/STOPPING_VERDICT.json").write_text(json.dumps(v, indent=2))


## 9. Pass B — does the budget rescale salvage anything?

The audit returned RETRAIN, so this is not the primary path. It is worth one checkpoint's
GPU time because it answers a separate, practical question: **can a per-language constant
convert a phoneme budget into the character budget this model was actually taught, well
enough to ship while the retrain runs?** Free if it works, and one run tells you.

In [ ]:
import os
if SCALE:
    os.environ["SCALE_PATH"] = str(SCALE)
    os.environ["BEST_CKPT"] = CKPTS[-1]
    print("rescale source:", SCALE)
else:
    print("no budget_scale.json attached — skipping pass B")


In [ ]:
import subprocess, sys
if SCALE:
    subprocess.run([sys.executable, "-m", "evaluation.phoneme_adherence_eval",
                    "--val_jsonl", str(VAL),
                    "--checkpoints", CKPTS[-1],
                    "--output_dir", "/kaggle/working/eval_rescaled",
                    "--mode", "length",
                    "--probe_sentences_per_lang", "25",
                    "--batch_size", "8",
                    "--budget_scale_json", str(SCALE),
                    "--wandb", "--wandb_required",
                    "--wandb_project", "indic-dubbing-v3",
                    "--wandb_entity", "nktthegreat-soccernet",
                    "--wandb_run_name", "02i-rescaled"], check=True)
else:
    print("no budget_scale.json — pass B skipped")


In [ ]:
import csv
from pathlib import Path

def load(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))     # materialise: a DictReader is single-pass

rows_a = load("/kaggle/working/eval_corrected/length_response.csv")
last = max(int(r["step"]) for r in rows_a)
a = {r["language"]: r for r in rows_a if int(r["step"]) == last}
b = {r["language"]: r for r in load("/kaggle/working/eval_rescaled/length_response.csv")}
print(f"{'lang':<6}{'probe slope A':>15}{'probe slope B':>15}{'delta':>9}")
for l in sorted(a):
    try:
        x, y = float(a[l]["length_slope_probe"]), float(b[l]["length_slope_probe"])
        print(f"{l:<6}{x:>15.3f}{y:>15.3f}{y-x:>+9.3f}")
    except Exception:
        pass
print("\nSave Version. Session C compares the retrained model against eval_corrected/.")
